In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)
sns.set(style="whitegrid")

import warnings
warnings.filterwarnings('ignore')

In [ ]:
#loading the data
try:
    from google.colab import files
    print("Please choose the file: INX_Future_Inc_Employee_Performance_CDS_Project2_Data_V1_8.xls")
    uploaded = files.upload()
    file_name = list(uploaded.keys())[0]
except ImportError:
   file_name = "INX_Future_Inc_Employee_Performance_CDS_Project2_Data_V1_8.xls"

print("Using file:", file_name)

In [ ]:
df = pd.read_excel(file_name)

df.head()


In [ ]:
## Checking shape of the dataset

df.shape

## The dataset has 1200 rows and 28 columns

In [ ]:
df.info()

## Dataset is divided into two data types namely:-

1. Intergers 19

2. objects 9

In [ ]:
## Checking for missing values

df.isnull().sum()

# No missing values in the dataset

In [ ]:
## checking of duplicates

df.duplicated().sum()

# No duplicated in the dataset

In [ ]:
## Checking data distribution in the dataset
df.describe()

## There is evidence of some statistical outliers.

## EXPLORATORY DATA ANALYSIS

In [ ]:
## Correlation heatmap aganist target variable (Perfomance rating)
plt.figure(figsize=(15,10))
sns.heatmap(df.corr(numeric_only=True), annot=True,cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

# Calculate correlation with 'PerformanceRating' for all numeric columns
corr_with_target = df.corr(numeric_only=True)['PerformanceRating'].sort_values(ascending=False)
corr_with_target = corr_with_target.drop('PerformanceRating')
print(corr_with_target.head(3))

### Correlation heatmap insights:

- **EmpEnvironmentSatisfaction** (0.4), **EmpLastSalaryHikePercent** (0.33) and **EmpWorkLifeBalance** (0.12) have positive correlation with performance Rating.



## BOXPLOT VISUALIZATION TO DETECT OUTLIERS

In [ ]:
numerical_cols = df.select_dtypes(include=np.number).columns

# Exclude 'PerformanceRating' as it's the target variable and its distribution has already been checked.
# Also exclude 'EmpEducationLevel', 'EmpEnvironmentSatisfaction', 'EmpJobInvolvement', 'EmpJobLevel',
# 'EmpJobSatisfaction', 'EmpRelationshipSatisfaction' as they are categorical encoded as numerical.
# These columns are ordinal and their 'outliers' might simply represent the extreme categories.

cols_to_plot = [col for col in numerical_cols if col not in ['PerformanceRating',
                                                          'EmpEducationLevel',
                                                          'EmpEnvironmentSatisfaction',
                                                          'EmpJobInvolvement',
                                                          'EmpJobLevel',
                                                          'EmpJobSatisfaction',
                                                          'EmpRelationshipSatisfaction']]

In [ ]:
# Box plot for NumCompaniesWorked
col = cols_to_plot[3]
plt.figure(figsize=(8, 6))
sns.boxplot(y=df[col])
plt.title(f'Boxplot of {col}')
plt.ylabel(col)
plt.show()

#### NumCompaniesWorked

*   **Interpretation**: The boxplot shows that a significant portion of employees have worked for a few companies (typically 1 to 4). The median is around 2 companies. There are individual data points (circles) above the upper whisker, indicating employees who have worked for a noticeably higher number of companies (up to 9). These are outliers, but they represent valid, albeit less common, career paths.

In [ ]:
# Box plot for TotalWorkExperienceInYears
col = cols_to_plot[5]
plt.figure(figsize=(8, 6))
sns.boxplot(y=df[col])
plt.title(f'Boxplot of {col}')
plt.ylabel(col)
plt.show()

#### TotalWorkExperienceInYears

*   **Interpretation**: The majority of employees have total work experience between 6 and 15 years, with a median around 10 years. There are numerous data points above the upper whisker, indicating employees with significantly more work experience (up to 40 years). These are outliers, but they are genuine data points reflecting long careers.

In [ ]:
# Box for TrainingTimesLastYear

col = cols_to_plot[6]
plt.figure(figsize=(8, 6))
sns.boxplot(y=df[col])
plt.title(f'Boxplot of {col}')
plt.ylabel(col)
plt.show()

#### TrainingTimesLastYear

*   **Interpretation**: Most employees had 2 or 3 training sessions last year, with the median at 3. There are outliers at 0, 5, and 6 training sessions. These values, while outside the main interquartile range, are valid counts for training participation.

In [ ]:
# Box Plot ExperienceYearsAtThisCompany
col = cols_to_plot[8]
plt.figure(figsize=(8, 6))
sns.boxplot(y=df[col])
plt.title(f'Boxplot of {col}')
plt.ylabel(col)
plt.show()

#### ExperienceYearsAtThisCompany

*   **Interpretation**: The boxplot shows a core group of employees with 3 to 10 years of experience at the current company, and a median around 5 years. A substantial number of outliers are present above the upper whisker, indicating employees with much longer tenure at the company (up to 40 years). These are meaningful data points representing long-serving employees.

In [ ]:
# Box Plot for ExperienceYearsInCurrentRole
col = cols_to_plot[9]
plt.figure(figsize=(8, 6))
sns.boxplot(y=df[col])
plt.title(f'Boxplot of {col}')
plt.ylabel(col)
plt.show()

#### ExperienceYearsInCurrentRole

*   **Interpretation**: Most employees have been in their current role for 2 to 7 years, with a median of 3 years. Outliers are observed above the upper whisker, indicating employees who have been in the same role for a significantly longer period (up to 18 years). These are valid observations, showing stability or lack of mobility within roles.

In [ ]:
# Box Plot for YearsSinceLastPromotion
col = cols_to_plot[10]
plt.figure(figsize=(8, 6))
sns.boxplot(y=df[col])
plt.title(f'Boxplot of {col}')
plt.ylabel(col)
plt.show()

#### YearsSinceLastPromotion

*   **Interpretation**: The majority of employees received a promotion within 0 to 3 years, with a median of 1 year. There are many outliers above the upper whisker, showing employees who have gone much longer without a promotion (up to 15 years). These are critical data points, as a long duration without promotion can be a significant factor in employee satisfaction and performance.

In [ ]:
#Box plot for YearsWithCurrManager
col = cols_to_plot[11]
plt.figure(figsize=(8, 6))
sns.boxplot(y=df[col])
plt.title(f'Boxplot of {col}')
plt.ylabel(col)
plt.show()

#### YearsWithCurrManager

*   **Interpretation**: Most employees have worked with their current manager for 2 to 7 years, with a median of 3 years. Outliers are visible above the upper whisker, representing employees who have been with the same manager for an extended period (up to 17 years). This can indicate either strong working relationships or a lack of career progression or management changes.



## Conclusion
Given that these outliers are likely valid observations and,  I will not drop the outliers since removing them might lead to a loss of potentially important information or patterns that the models could learn from.

In [ ]:
## Employee Perfomance rating distribution

plt.figure(figsize=(5,4))
sns.countplot(x='PerformanceRating', data=df, palette='viridis')
plt.title('Distribution of Performance Rating')
plt.xlabel('Performance Rating (2=Low, 3=Good, 4=Excellent)')
plt.ylabel('Number of Employees')
plt.show()

df['PerformanceRating'].value_counts(normalize=True) * 100


# It is evident that 73% of the employees were rated 3, 16%  of the employees were rated 2, while 11% of the employees were rated 4.

In [ ]:
# Heatmap for average performance rating by department and job role
plt.figure(figsize=(18, 10))
sns.heatmap(dept_role_perf, annot=True, cmap='YlGnBu', fmt=".2f", linewidths=.5)
plt.title('Average Performance Rating Heatmap by Department and Job Role')
plt.xlabel('Job Role')
plt.ylabel('Department')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Interpretation from the Heatmap:

1.  **Overall High-Performing Departments**: The **Development** and **Data Science** departments consistently show higher average performance ratings across their various job roles. For example, 'Business Analyst' (3.19) and 'Developer' (3.11) in Development, and 'Data Scientist' (3.05) in Data Science, exhibit strong performance.

2.  **Lower-Performing Departments & Key Gaps**: The **Finance** department stands out with generally lower performance ratings across its roles, with 'Finance Manager' (2.78) being an example. The **Sales** department also shows some areas of concern with 'Sales Executive' (2.86) and 'Sales Representative' (2.87) having moderate to lower average ratings compared to other high-performing roles.

3.  **Variability within Departments**: Even within a department, performance can vary. In **Research & Development**, while 'Manager' (3.25) and 'Manufacturing Director' (2.97) perform well, 'Laboratory Technician' (2.85) shows a slightly lower average.

4.  **Role-Specific Strengths and Weaknesses**: Certain roles, like 'Business Analyst', 'Developer', 'Technical Lead', and 'Data Scientist', generally maintain high performance wherever they are present. Conversely, roles in departments like Finance and parts of Sales consistently underperform.



# Recommendations:

*Targeted Interventions*:
- The identified lower-performing departments (Finance, and certain roles within Sales) clearly indicate a need for targeted interventions. Instead of blanket solutions, HR can focus on understanding the specific challenges faced by 'Finance Managers' or 'Sales Representatives' through focus groups or interviews.

*Role-Specific Development*:
- The varying performance across job roles highlights the importance of tailored career pathways and development. Learning from the success of 'Business Analysts' in Development could inform training programs for other roles, while lower-performing roles might benefit from re-evaluation of expectations or additional skill-building.

*Environmental Factors*:
- The performance differences might be linked to underlying environmental satisfaction factors. Investigating if these lower-performing departments or roles face unique stressors could uncover root causes.


## FEATURE ENGINEERING

- EmpNumber has no predictive value, will have to drop the column.
- I Will have to convert categorical columns i.e Gender, EducationBackground, MaritalStatus, EmpDepartment, EmpJobRole, BusinessTravelFrequency, 'OverTime', Attrition using Label Encoder.



In [ ]:
data = df.copy()

# Drop EmpNumber column as it has no predictive value
data.drop('EmpNumber', axis=1, inplace=True)

# Columns to be Label Encoded (for features where ordinality might be tolerated or for high cardinality)
le_cols = ['EmpDepartment', 'EmpJobRole']

# Columns to be One-Hot Encoded (for nominal categorical features)
ohe_cols = ['EducationBackground', 'Gender', 'MaritalStatus', 'BusinessTravelFrequency', 'OverTime', 'Attrition']

# Apply Label Encoding
encoder_le = {}
for col in le_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    encoder_le[col] = le

# Apply One-Hot Encoding
# Using handle_unknown='ignore' and drop_first=True for better generalization and avoiding multicollinearity with linear models later.
from sklearn.preprocessing import OneHotEncoder

encoder_ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first')
ohe_transformed_data = encoder_ohe.fit_transform(data[ohe_cols])

# Get feature names for one-hot encoded columns
ohe_feature_names = encoder_ohe.get_feature_names_out(ohe_cols)
ohe_df = pd.DataFrame(ohe_transformed_data, columns=ohe_feature_names, index=data.index)

# Drop original one-hot encoded columns from data and concatenate the new OHE DataFrame
data = data.drop(columns=ohe_cols)
data = pd.concat([data, ohe_df], axis=1)

display(data.head())

In [ ]:
# Split feature X and target y

X = data.drop('PerformanceRating', axis=1)
y = data['PerformanceRating']

In [ ]:
#Train,Test and Split

X_train, X_test,y_train,y_test = train_test_split(X,y, test_size=0.2, random_state=42, stratify = y)
# stratify=y keeps the same proportion of each performance rating in both sets,important because the classes are imbalanced

## RANDOM FOREST CLASSIFIER

I used a Random Forest Classifier:- it builds many decision trees on random subsets of the data and features, then allows them "vote" on the final prediction.

-It needs almost no data scaling/normalization.

-It handles both numeric and encoded categorical data well.

-It gives us feature importance literally telling us which factors matter most

In [ ]:
model = RandomForestClassifier (
    n_estimators=100,           #Number of trees in the forest
    max_depth=None,             #Allows trees to grow fully
    random_state=42,
    class_weight = 'balanced'   # helps with the imbalanced classes
)
model.fit(X_train, y_train)


In [ ]:
#Model Evaluation

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")



*Performance Rating 2 (Low): F1-score = 0.88

-  This indicates a good balance between precision and recall for identifying 'Low' performing employees. The model is reasonably effective at correctly labeling low performers without making too many false positive or false negative errors for this class.

*Performance Rating 3 (Good/Average): F1-score = 0.96

-  This is a very high F1-score, suggesting that the model is excellent at identifying 'Good/Average' performing employees. Given that this is the majority class, the model performs exceptionally well in both correctly identifying these employees and avoiding misclassifying others as 'Good/Average'.

*Performance Rating 4 (Excellent): F1-score = 0.84

-  This F1-score, while lower than for the 'Good/Average' class, still represents a decent performance for the 'Excellent' class. The model shows a reasonable balance in identifying excellent performers, though it has some room For improvement, note that its higher recall (0.73) and perfect precision (1.00) means it did not misclassify any non-excellent employee as excellent but missed some actual excellent employees.


*Accuracy = 0.93 (93%)

- The model correctly predicted the PerformanceRating for 93% of the employees in the test dataset.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
cm = ConfusionMatrixDisplay.from_estimator(
    model, X_test, y_test, cmap='Oranges', ax=ax,
    display_labels=['Low', 'Good/Average', 'Excellent']
)
plt.title('Confusion Matrix')
plt.show()

## Key observations from this matrix:
- The model is very good at identifying 'Good/Average' performers. It has some difficulty distinguishing 'Low' from 'Good/Average' and 'Excellent' from 'Good/Average'. Notably, it misclassifies a notable portion of 'Excellent' employees as 'Good/Average'.

- 32 employees correctly identified as low performance, 173 employees correctly identified as Good/Average perfomance while 19 employees identified as excellent perfomance.
- 7 employees where were misclassified as Good/Average , 2 employees were Misclassified as low while 7 employees were misclassified as Good/Average.

In [ ]:
# Cross-validation to check the stability of the model

cv_scores = cross_val_score(model, X_train, y_train, cv=5)
print(f"Cross-validation scores: {cv_scores}")
print(f"Average CV accuracy: {cv_scores.mean():.2f}")

- Individual Fold Scores: [0.91145833 0.93229167 0.90104167 0.921875   0.94270833] represents the accuracy of the model on each of the five folds of the training data. This indicates that the model's performance is quite consistent across different partitions of the data.

- Average CV Accuracy of 92% on the training data suggests that the model is robust and not overly sensitive to the specific training data split. This is very close to the 93% accuracy observed on the model test set, which further reinforces that the model is not overfitting to the training data.

## Feature Importance

This will tell the most valuable features the model relied on the prediction

In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(10,6))
importance.head(5).sort_values().plot(kind='barh', color='orange')
plt.title('Top 5 Most Important Features for Predicting Performance')
plt.ylabel('Importance')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

importance.head(5)

*The top 3 factors that are affecting employee performance are:-*
- Employee's Last Salary Hike,
- Employee's Environment Satisfaction,
- Employee's last promotion.

# GRADIENT BOOSTING CLASSIFIER

I will be using the gradient boosting classifier because of several reasons:

- **High Accuracy**: It creates a series of decision trees, where each tree fixes mistakes made by the previous one. This approach leads to better predictions and strong results with data.

- **Handles Complex Relationships**: It can find different factors that affect performance.

- **Handles Imbalanced Classes**: It can manage class imbalances by using techniques like sampling or changing loss functions.

In [ ]:
# Initialize and train the Gradient Boosting Classifier
gbc_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gbc_model.fit(X_train, y_train)


## Model Evaluation

In [ ]:
# Predict on the test set
y_pred_gbc = gbc_model.predict(X_test)

# Evaluate the model
print("Classification Report for Gradient Boosting Classifier:")
print(classification_report(y_test, y_pred_gbc))

accuracy_gbc = accuracy_score(y_test, y_pred_gbc)
print(f"Accuracy: {accuracy_gbc:.2f}")



*Performance Rating 2 (Low): F1-score = 0.88*

- This indicates a strong balance between precision and recall for the 'Low' performance category. The model is effective at correctly identifying employees with low performance and has a relatively low rate of misclassification for this group.

*Performance Rating 3 (Good/Average): F1-score = 0.96*

- This is an excellent F1-score, showing that the model is highly accurate in identifying employees with 'Good/Average' performance. Given that this is the majority class, the model performs exceptionally well in both correctly classifying these employees and avoiding false positives or false negatives.

*Performance Rating 4 (Excellent): F1-score = 0.86*

- This F1-score suggests good performance for the 'Excellent' class. While slightly lower than the 'Good/Average' class, it demonstrates a robust ability to identify excellent performers, balancing the need to find all of them (recall) with the need to avoid incorrectly labeling others as excellent (precision).

*Accuracy = 0.94 (94%)*

- The model correctly pre- dicted the PerformanceRating for 94% of the employees in the test dataset, indicating its overall effectiveness.

In [ ]:
# Plot Confusion Matrix for Gradient Boosting
fig_gbc, ax_gbc = plt.subplots(figsize=(8, 6))
cm_gbc = ConfusionMatrixDisplay.from_estimator(
    gbc_model, X_test, y_test, cmap='Greens', ax=ax_gbc,
    display_labels=['Low', 'Good/Average', 'Excellent']
)
plt.title('Confusion Matrix for Gradient Boosting Classifier')
plt.show()

## Key observations from this confusion matrix:

Similar to the Random Forest model, the Gradient Boosting model also performs very well in identifying 'Good/Average' performance.
It misclassifies some 'Low' performers as 'Good/Average' and vice-versa.
It struggles a bit with 'Excellent' performers, misclassifying 5 of them as 'Good/Average'. It also has one instance where a 'Good/Average' performer was incorrectly classified as 'Excellent' (False Positive for 'Excellent').

In [ ]:
#Cross validation to evaluate perform of the GBC perfomance.
cv_scores_gbc = cross_val_score(gbc_model, X_train, y_train, cv=5)
print(f"Cross-validation scores for Gradient Boosting Classifier: {cv_scores_gbc}")
print(f"Average CV accuracy for Gradient Boosting Classifier: {cv_scores_gbc.mean():.2f}")

*Individual Cross-validation scores: [0.91666667 0.94270833 0.94791667 0.91145833 0.9375]*

*Average CV accuracy: 0.93*
- This shows that the model is also quite stable, with an average accuracy of 93% across the different folds of the training data.

## Feature Importance

In [ ]:
importance_gbc = pd.Series(gbc_model.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(10,6))
importance_gbc.head(5).sort_values().plot(kind='barh', color='orange')
plt.title('Top 5 Most Important Features for Predicting Performance')
plt.ylabel('Importance')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

importance_gbc.head(5)

*Similar to Random forest, the top 3 factors that affecting employee performance are:-*

- Employee's environment satisfaction,
- Employee's last salary hike,
- Employee's last promotion.

## Logistic Regression Classifier (Baseline Linear Model)

To provide a baseline for comparison, we will now train and evaluate a Logistic Regression model. This linear model will help us understand the performance improvement offered by more complex models like Random Forest and Gradient Boosting.

In [ ]:
# Initialize and train the Logistic Regression Classifier
# Using 'liblinear' solver for better performance with 'l1' regularization and 'balanced' class_weight
lr_model = LogisticRegression(
    solver='liblinear',
    penalty='l1', # L1 regularization can help with feature selection
    class_weight='balanced', # Handle imbalanced classes
    random_state=42,
    max_iter=1000 # Increase max_iter for convergence
)

lr_model.fit(X_train, y_train)

In [ ]:
# Model Evaluation

# Predict on the test set
y_pred_lr = lr_model.predict(X_test)

# Evaluate the model
print("Classification Report for Logistic Regression Classifier:")

print(classification_report(y_test, y_pred_lr))

accuracy_lr = accuracy_score(y_test, y_pred_lr)
print(f"Accuracy: {accuracy_lr:.2f}")

*Performance Rating 2 (Low): F1-score = 0.55*

- This indicates a moderate balance between precision and recall for identifying 'Low' performing employees. The model has some difficulty in correctly identifying this class without making a significant number of false positives or false negatives.

*Performance Rating 3 (Good/Average): F1-score = 0.87*

- This is a good F1-score, suggesting that the model is quite effective at identifying 'Good/Average' performing employees, balancing precision and recall well for this majority class.

*Performance Rating 4 (Excellent): F1-score = 0.70*

- This F1-score indicates a reasonable performance for the 'Excellent' class, but it's lower than the 'Good/Average' class. This means there's some trade-off between correctly identifying all excellent performers and avoiding
misclassifying others as excellent.

*Overall Accuracy: 0.80 (80%)*

- This model correctly predicted the PerformanceRating for 80% of the employees in the test dataset. This is a general measure of how well the model performed across all classes.

In [ ]:
# Plot Confusion Matrix for Logistic Regression
fig_lr, ax_lr = plt.subplots(figsize=(8, 6))
cm_lr = ConfusionMatrixDisplay.from_estimator(
    lr_model, X_test, y_test, cmap='Oranges', ax=ax_lr,
    display_labels=['Low', 'Good/Average', 'Excellent']
)
plt.title('Confusion Matrix for Logistic Regression Classifier')
plt.show()

# Key observation
This confusion matrix shows its performance in classifying employees into 'Low', 'Good/Average', and 'Excellent' performance ratings:

*Low Performance:*

- 23 employees were correctly identified as 'Low'. 13 were misclassified as 'Good/Average', and 3 as 'Excellent'.

*Good/Average Performance:*

- 150 employees were correctly identified as 'Good/Average'. However, 19 were misclassified as 'Low', and 6 as 'Excellent'.

*Excellent Performance:*

- 19 employees were correctly identified as 'Excellent'. 2 were misclassified as 'Low', and 5 as 'Good/Average'.

Overall, the model performs best at identifying 'Good/Average' performers but has some difficulty distinguishing between 'Low' and 'Good/Average', and also misclassifies some 'Excellent' performers

In [ ]:
# Cross-validation to evaluate performance of the Logistic Regression model
cv_scores_lr = cross_val_score(lr_model, X_train, y_train, cv=5)
print(f"Cross-validation scores for Logistic Regression Classifier: {cv_scores_lr}")
print(f"Average CV accuracy for Logistic Regression Classifier: {cv_scores_lr.mean():.2f}")

*Individual Cross-validation scores: [0.77604167 0.765625   0.83854167 0.77083333 0.78125   ]*

*Average CV accuracy: 0.79*
- This shows that the model is also quite stable, with an average accuracy of 80% across the different folds of the training data.

## Feature Importance.

In [ ]:
importance_lr = pd.Series(np.sum(np.abs(lr_model.coef_), axis=0), index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10,6))
importance_lr.head(5).sort_values().plot(kind='barh', color='orange')
plt.title('Top 5 Most Important Features for Predicting Performance (Logistic Regression)')
plt.ylabel('Feature')
plt.xlabel('Absolute Coefficient Sum')
plt.tight_layout()
plt.show()

print("Top 5 Logistic Regression Features:")
print(importance_lr.head(5))

*The top 3 factors that are affecting employee performance are:-*

- Employee's Environment Satisfaction,
- Employee worklife balance
- Education background in marketing

# Final Model Selection

In [ ]:
models = ['Random Forest', 'Gradient Boosting', 'Logistic Regression']
f1_scores = [0.93, 0.94, 0.80]

plt.figure(figsize=(6, 4))
plt.barh(models, f1_scores, color='orange')
plt.title('Achieved Weighted F1 Score for All Models')
plt.xlabel('Weighted F1 Score')
plt.ylabel('Model')
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

To select the final model, will compare the performance of the three trained classifiers (Random Forest, Gradient Boosting, and Logistic Regression) using the *weighted F1-score*. This metric is particularly suitable given the class imbalance in our target variable (PerformanceRating), as it provides a balanced measure of precision and recall across all classes.

*   **Gradient Boosting Classifier**: Achieved a weighted F1-score of **0.94**.
*   **Random Forest Classifier**: Achieved a weighted F1-score of **0.93**.
*   **Logistic Regression Classifier**: Achieved a weighted F1-score of **0.80**.

Based on these results, the **Gradient Boosting Classifier** demonstrates the best overall performance with the highest weighted F1-score. Therefore, it is selected as the final model for predicting employee performance ratings.

In [ ]:
# Save the trained model to a file
import joblib

# Define final_model and model_name
final_model = gbc_model
model_name = "Gradient Boosting Classifier"

joblib.dump(final_model, 'employee_performance_model.pkl')
print(f"The {model_name} has been saved as 'employee_performance_model.pkl'")

# Recommendations to Improve Employee Performance

1. Review the salary hike structure to ensure raises are fair, timely, and clearly tied to contribution, this is one of the strongest performance levers available to management.
2. Invest in workplace environment initiatives (tools, workspace, manager support, team culture surveys) since environment satisfaction is the single strongest controllable driver of performance identified in this analysis.
3. Create a clearer promotion / career-growth pathway. Employees who have gone a long time without promotion should be flagged for a career conversation, or a formal review, rather than being overlooked.
4. Investigate the Sales and Finance departments specifically; run focus groups or exit interviews to understand what is different there (targets, incentive structure, travel demands, manager quality) versus higher-performing departments.
5. Rotate roles / responsibilities periodically for employees with long tenure in the same role or under the same manager, to counter performance plateaus.
6. Use this model as a decision-support tool, not a replacement for manager judgment. It can help flag employees at risk of a low rating early, so HR/managers can intervene (training, support, conversation) before the formal review, making the process feel more proactive and fair rather than purely retrospective.


